# Diagnóstico aislado de Sentinel-3

Prueba catálogo, Process API y Statistical API con una muestra pequeña. No ejecuta el pipeline general ni descarga productos completos.

## 1. Configuración y credenciales

In [ ]:
from pathlib import Path
import os

# Completa estos campos o define SH_CLIENT_ID y SH_CLIENT_SECRET en el entorno.
SH_CLIENT_ID = os.getenv("SH_CLIENT_ID", "")
SH_CLIENT_SECRET = os.getenv("SH_CLIENT_SECRET", "")
SH_CLIENT_ID = SH_CLIENT_ID or os.getenv("SH_CLIENT_ID", "")
SH_CLIENT_SECRET = SH_CLIENT_SECRET or os.getenv("SH_CLIENT_SECRET", "")

CENTER_LAT = 37.3560356
CENTER_LON = -5.9804827
HALF_SIDE_M = 750
PROJECTED_CRS = "EPSG:25830"
S3_REQUEST_CRS = "EPSG:32630"  # CRS UTM esperado por Sentinel-3 SLSTR en Sevilla
DATE_START = "2024-07-01"
DATE_END = "2024-07-07"
TARGET_HOUR_UTC = 10.5
TEST_SCENES = 3
REQUEST_TIMEOUT_S = 90

OUTPUT_DIR = Path.cwd() / "diagnostico_sentinel3"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Preparación, token y ROI

In [ ]:
import json
from time import perf_counter

import pandas as pd
import requests
from pyproj import Transformer

from tools.sentinelhub_s3 import (
    S3_COLLECTION, S3_STATISTICS_URL, aligned_grid_cells, aligned_output_grid,
    catalogue_rows, choose_s3_products, process_request_payload,
    statistics_request_payload, statistics_response_rows,
)

if not SH_CLIENT_ID or not SH_CLIENT_SECRET:
    raise ValueError("Completa SH_CLIENT_ID y SH_CLIENT_SECRET en la cabecera.")

token_response = requests.post(
    "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token",
    data={"grant_type": "client_credentials", "client_id": SH_CLIENT_ID, "client_secret": SH_CLIENT_SECRET},
    timeout=30,
)
token_response.raise_for_status()
token = token_response.json()["access_token"]
headers = {"Authorization": f"Bearer {token}"}

to_projected = Transformer.from_crs("EPSG:4326", PROJECTED_CRS, always_xy=True)
cx, cy = to_projected.transform(CENTER_LON, CENTER_LAT)
roi_bounds = [cx - HALF_SIDE_M, cy - HALF_SIDE_M, cx + HALF_SIDE_M, cy + HALF_SIDE_M]
to_s3_request = Transformer.from_crs(PROJECTED_CRS, S3_REQUEST_CRS, always_xy=True)
s3_corners = [
    to_s3_request.transform(x, y)
    for x in (roi_bounds[0], roi_bounds[2])
    for y in (roi_bounds[1], roi_bounds[3])
]
s3_roi_bounds = [
    min(point[0] for point in s3_corners), min(point[1] for point in s3_corners),
    max(point[0] for point in s3_corners), max(point[1] for point in s3_corners),
]
to_wgs84 = Transformer.from_crs(PROJECTED_CRS, "EPSG:4326", always_xy=True)
min_lon, min_lat = to_wgs84.transform(roi_bounds[0], roi_bounds[1])
max_lon, max_lat = to_wgs84.transform(roi_bounds[2], roi_bounds[3])

def timed_post(label, url, payload, timeout=REQUEST_TIMEOUT_S):
    started = perf_counter()
    try:
        response = requests.post(url, headers=headers, json=payload, timeout=timeout)
        elapsed = perf_counter() - started
        record = {
            "test": label, "status": response.status_code, "seconds": elapsed,
            "bytes": len(response.content),
            "retry_after": response.headers.get("Retry-After"),
            "request_id": response.headers.get("x-request-id") or response.headers.get("x-correlation-id"),
            "content_type": response.headers.get("Content-Type"),
            "error_excerpt": "" if response.ok else response.text[:500],
        }
        return response, record
    except requests.Timeout:
        return None, {"test": label, "status": "TIMEOUT", "seconds": perf_counter()-started, "bytes": 0,
                      "retry_after": None, "request_id": None, "content_type": None,
                      "error_excerpt": f"Sin respuesta en {timeout} segundos"}
    except requests.RequestException as exc:
        return None, {"test": label, "status": type(exc).__name__, "seconds": perf_counter()-started, "bytes": 0,
                      "retry_after": None, "request_id": None, "content_type": None,
                      "error_excerpt": str(exc)[:500]}

print("Token y ROI preparados; los secretos no se guardan en las salidas.")

## 3. Catálogo y selección temporal

In [ ]:
catalog_payload = {
    "collections": [S3_COLLECTION],
    "bbox": [min_lon, min_lat, max_lon, max_lat],
    "datetime": f"{DATE_START}T00:00:00Z/{DATE_END}T23:59:59Z",
    "limit": 100,
}
catalog_response, catalog_metric = timed_post(
    "catalog", "https://sh.dataspace.copernicus.eu/catalog/v1/search", catalog_payload
)
if catalog_response is None or not catalog_response.ok:
    raise RuntimeError(f"Catálogo fallido: {catalog_metric}")
products = catalogue_rows(catalog_response.json().get("features", []))
selected, available_days = choose_s3_products(
    products, morning_start_hour=7, morning_end_hour=13, target_hour=TARGET_HOUR_UTC
)
selected.to_csv(OUTPUT_DIR / "catalogo_seleccionado.csv", index=False)
print(f"{len(products)} escenas; {available_days} días; {len(selected)} adquisiciones diarias seleccionadas.")
display(selected[["Name", "acquisition_dt_utc", "hour_distance"]].head(10))

## 4. Process API: pocas escenas, sin reintentos ocultos

In [ ]:
process_bounds, width, height = aligned_output_grid(s3_roi_bounds, 1000)
process_metrics = []
for product in list(selected.head(TEST_SCENES).itertuples()):
    payload = process_request_payload(process_bounds, S3_REQUEST_CRS, width, height, product.acquisition_dt_utc)
    response, metric = timed_post(f"process:{product.Name}", "https://sh.dataspace.copernicus.eu/process/v1", payload)
    process_metrics.append(metric)
    if response is not None and response.ok:
        (OUTPUT_DIR / f"process_{len(process_metrics):02d}.tif").write_bytes(response.content)
    if response is not None:
        response.close()
process_metrics = pd.DataFrame(process_metrics)
process_metrics.to_csv(OUTPUT_DIR / "resultados_process.csv", index=False)
display(process_metrics)

## 5. Statistical API: siete días en una petición

In [ ]:
cell = aligned_grid_cells(process_bounds, width, height)[0]
stats_payload = statistics_request_payload(
    cell["bounds"], S3_REQUEST_CRS, DATE_START, pd.Timestamp(DATE_END) + pd.Timedelta(days=1),
    TARGET_HOUR_UTC, resolution_m=1000,
)
stats_response, stats_metric = timed_post("statistics:daily-batch", S3_STATISTICS_URL, stats_payload)
stats_rows = []
if stats_response is not None and stats_response.ok:
    stats_json = stats_response.json()
    (OUTPUT_DIR / "respuesta_statistics.json").write_text(json.dumps(stats_json, indent=2), encoding="utf-8")
    stats_rows = statistics_response_rows(stats_json)
    pd.DataFrame(stats_rows).to_csv(OUTPUT_DIR / "valores_statistics.csv", index=False)
if stats_response is not None:
    stats_response.close()
display(pd.DataFrame([stats_metric]))
display(pd.DataFrame(stats_rows).head())

## 6. Diagnóstico automático

In [ ]:
metrics = pd.concat([
    pd.DataFrame([catalog_metric]), process_metrics, pd.DataFrame([stats_metric])
], ignore_index=True)
metrics.to_csv(OUTPUT_DIR / "resultados_diagnostico.csv", index=False)

statuses = metrics["status"].astype(str)
error_text = metrics["error_excerpt"].fillna("").astype(str).str.cat(sep=" ")
if "Invalid envelope CRS" in error_text:
    diagnosis = "CONFIGURACIÓN: el CRS enviado no coincide con el requerido por Sentinel-3 SLSTR."
elif statuses.eq("429").any():
    diagnosis = "LIMITACIÓN: Sentinel Hub responde 429; hay exceso de frecuencia o cuota temporal."
elif statuses.eq("TIMEOUT").any():
    diagnosis = "TIMEOUT: Process API no responde dentro del límite; el pipeline anterior podía esperar cinco minutos por escena."
elif statuses.isin(["401", "403"]).any():
    diagnosis = "AUTORIZACIÓN: las credenciales son válidas para el token, pero la API rechaza la operación o colección."
elif statuses.str.startswith("5").any():
    diagnosis = "SERVIDOR: Sentinel Hub está devolviendo errores temporales 5xx."
elif process_metrics["status"].astype(str).eq("200").all() and str(stats_metric["status"]) == "200":
    process_total = process_metrics["seconds"].sum()
    diagnosis = (
        f"OPERATIVO: Process API necesitó {process_total:.1f}s para {len(process_metrics)} escenas; "
        f"Statistical API resolvió el lote diario en {stats_metric['seconds']:.1f}s."
    )
else:
    diagnosis = "FALLO MIXTO: revisa status, error_excerpt y request_id en resultados_diagnostico.csv."

report = [
    "DIAGNÓSTICO SENTINEL-3", "", diagnosis, "",
    f"Escenas Process probadas: {len(process_metrics)}",
    f"Días devueltos por Statistical API: {len(stats_rows)}",
    f"CRS científico del proyecto: {PROJECTED_CRS}; CRS de petición Sentinel-3: {S3_REQUEST_CRS}.",
    "Los secretos no se incluyen en este informe.",
]
(OUTPUT_DIR / "diagnostico.txt").write_text("\n".join(report) + "\n", encoding="utf-8")
print(diagnosis)
print("Resultados:", OUTPUT_DIR.resolve())